# EarningsLens — Stage 6: Cross-Quarter Commitment Tracking

Stage 6 is the intelligence layer that differentiates EarningsLens from every existing financial data product. It answers the question no Bloomberg or FactSet tool currently answers: **did management actually deliver on what they said last quarter?**

For each company in the 2023–2025 corpus, commitments from different quarters are matched against each other using sentence embeddings and cosine similarity. When the same commitment is found rephrased across quarters, the extracted `value` slots are compared to assign a status:

- **Delivered** — the later quarter's value met or exceeded the earlier commitment
- **Raised** — guidance was revised upward
- **Missed** — the later quarter's value fell short of the commitment
- **Withdrawn** — the commitment disappeared in subsequent quarters without a follow-up
- **Pending** — the most recent quarter's commitment with no subsequent data yet

### Architecture

**Step 1 — Embedding:** Every commitment sentence is encoded with `all-MiniLM-L6-v2`, a lightweight but highly accurate sentence embedding model fine-tuned for semantic similarity tasks.

**Step 2 — Matching:** Within each company, commitments from different quarters are compared by cosine similarity. Pairs with similarity ≥ 0.70 and the same normalised metric label are treated as the same commitment rephrased. This threshold was chosen based on the cross-quarter guidance literature — below 0.70 the matches become too noisy; above 0.85 too many genuine rephrasings are missed.

**Step 3 — Status assignment:** Matched pairs have their `value` slots parsed into numeric form. A value parser handles the common financial figure formats — dollar amounts, percentages, ranges — and falls back to string comparison when parsing fails.

**Step 4 — Output:** The `status` and `matched_commitment_id` columns are written to `fls_2023_2025.parquet`. This file is the complete pipeline output for the 2023–2025 scope and feeds directly into Stage 7 (sentiment analysis), Stage 8 (RAG), Stage 9 (agentic layer), and Stage 10 (dashboard).

### Scope

Tracking is run on commitments where at least one of `metric`, `value`, or `timeframe` was successfully extracted in Stage 4 — unslotted sentences are retained in the dataset with `status = Pending` but are not candidates for cross-quarter matching.

## 0. Dependencies

`sentence-transformers` provides the `all-MiniLM-L6-v2` embedding model. `scikit-learn` provides the cosine similarity computation. Both are lightweight and install quickly.

In [ ]:
# %pip install -q sentence-transformers scikit-learn pandas tqdm

## 0.1 Imports & Logging

In [ ]:
# Standard library
import re
import os
import logging
from pathlib import Path
from itertools import combinations
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
from difflib import SequenceMatcher

# Data
import numpy as np
import pandas as pd
from tqdm import tqdm

# Embeddings and similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO,
)
log = logging.getLogger("earningslens.stage6")

c:\Users\sidsu\Desktop\Sem 4\NLP\NLPvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
if not DATABASE_URL:
    raise EnvironmentError("DATABASE_URL not found — check your .env file")

## 1. Configuration

`SIMILARITY_THRESHOLD` of 0.70 is the cosine similarity cutoff for treating two commitment sentences as the same commitment rephrased. This value was selected based on the cross-quarter guidance matching literature — it balances recall (catching genuine rephrasings) against precision (avoiding false matches on thematically similar but distinct commitments).

`MIN_SLOTS_FOR_MATCHING` ensures only commitments with at least one extracted slot enter the matching pool. Unslotted sentences are retained in the output with `status = Pending` but cannot be meaningfully tracked across quarters without a metric anchor.

In [ ]:
# Paths
INPUT_PATH  = Path("fls_2023_2025.parquet")
OUTPUT_PATH = Path("fls_2023_2025.parquet")   # overwritten in-place

# Embedding model
EMBEDDING_MODEL      = "sentence-transformers/all-MiniLM-L6-v2"

# Matching parameters
SIMILARITY_THRESHOLD   = 0.70   # cosine similarity cutoff for commitment matching
MIN_SLOTS_FOR_MATCHING = 2      # minimum non-null slots required to enter matching pool

# Quarter ordering for cross-quarter comparison
# Used to determine which commitment is 'earlier' and which is 'later'
def quarter_key(year: int, quarter: int) -> int:
    """Convert year + quarter to a sortable integer. e.g. 2024 Q3 -> 20243"""
    return year * 10 + quarter

print("Configuration set")
print(f"  Embedding model      : {EMBEDDING_MODEL}")
print(f"  Similarity threshold : {SIMILARITY_THRESHOLD}")

Configuration set
  Embedding model      : sentence-transformers/all-MiniLM-L6-v2
  Similarity threshold : 0.7


## 2. Load Data

`fls_2023_2025.parquet` is the output of Stage 5 — 269,679 Specific FLS sentences with `metric`, `value`, `timeframe`, and `hedge_score` columns populated for the 2023–2025 scope. A `commitment_id` column is assigned here as a stable integer identifier for each row, used as the join key when writing matched pairs back to the DataFrame.

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"{INPUT_PATH} not found. "
        "Ensure Stage 5 completed and saved fls_2023_2025.parquet."
    )

In [ ]:
fls = pd.read_parquet(INPUT_PATH)
log.info("Loaded %d rows from %s", len(fls), INPUT_PATH)

2026-05-07 03:29:18,379 [INFO] Loaded 92458 rows from fls_2023_2025.parquet


In [ ]:
# Assign a stable integer commitment_id if not already present
if "commitment_id" not in fls.columns:
    fls["commitment_id"] = range(len(fls))

In [ ]:
# Initialise status and matched_commitment_id columns
if "status" not in fls.columns:
    fls["status"] = "Pending"
if "matched_commitment_id" not in fls.columns:
    fls["matched_commitment_id"] = None

print(f"Total commitments        : {len(fls):,}")
print(f"Unique tickers           : {fls['ticker'].nunique():,}")
print(f"Unique transcript IDs    : {fls['transcript_id'].nunique():,}")
print(f"\nYear / quarter distribution:")
print(fls.groupby(["year", "quarter"]).size().to_string())

Total commitments        : 92,458
Unique tickers           : 531
Unique transcript IDs    : 4,664

Year / quarter distribution:
year  quarter
2023  1           9303
      2          10034
      3          10167
      4          13128
2024  1           5123
      2           9438
      3          10155
      4          13066
2025  1           8526
      2           1984
      3           1078
      4            456


## 3. Filter Matchable Commitments

Only commitments with at least one non-null slot (`metric`, `value`, or `timeframe`) enter the matching pool. Unslotted commitments — qualitative forward-looking statements where the LLM could not identify a concrete figure — are retained in `fls` with `status = Pending` but excluded from embedding and similarity computation.

The `quarter_key` column provides a sortable integer for ordering commitments chronologically within each company.

In [ ]:
# Count non-null slots per row
fls["slot_count"] = (
    fls["metric"].notna().astype(int) +
    fls["value"].notna().astype(int) +
    fls["timeframe"].notna().astype(int)
)

In [ ]:
# Add sortable quarter key
fls["quarter_key"] = fls.apply(
    lambda r: quarter_key(int(r["year"]), int(r["quarter"])), axis=1
)

In [ ]:
matchable = fls[fls["slot_count"] >= 2].copy()
print(f"matchable rows: {len(matchable):,}")

matchable rows: 38,972


## 4. Load Embedding Model

`all-MiniLM-L6-v2` is a 22M parameter sentence transformer fine-tuned for semantic similarity. It produces 384-dimensional embeddings and runs efficiently on both CPU and GPU. At ~14k sentences/second on GPU, embedding 269k matchable sentences takes approximately 20 seconds.

In [ ]:
log.info("Loading embedding model: %s", EMBEDDING_MODEL)

embedder = SentenceTransformer(EMBEDDING_MODEL)

# Use GPU if available
import torch
if torch.cuda.is_available():
    embedder = embedder.to(torch.device("cuda"))
    log.info("Embedder on CUDA: %s", torch.cuda.get_device_name(0))
else:
    log.info("Embedder on CPU")

print("\u2713 Embedding model loaded")

2026-05-07 03:29:19,495 [INFO] Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
2026-05-07 03:29:19,501 [INFO] No device provided, using cuda:0
2026-05-07 03:29:20,099 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 03:29:20,120 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-07 03:29:20,361 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 03:29:20,385 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-07 03:29:20,392 [INFO] Loadin

✓ Embedding model loaded


## 5. Compute Embeddings

All matchable commitment sentences are embedded in a single batched pass. The embedder uses the raw `sentence` text rather than the extracted `metric` slot — the full sentence carries richer semantic context for matching rephrased commitments than a normalised slot string would. Embeddings are normalised to unit length before similarity computation so that cosine similarity reduces to a dot product.

In [ ]:
sentences_to_embed = matchable["sentence"].tolist()

In [ ]:
print(f"sentences_to_embed length : {len(sentences_to_embed)}")
print(f"fls length                : {len(fls)}")

# If you have a checkpoint file
from pathlib import Path
ckpt = Path("embeddings_checkpoint.parquet")  # or whatever it's named
if ckpt.exists():
    existing = pd.read_parquet(ckpt)
    print(f"Checkpoint rows           : {len(existing):,}")
    print(f"Checkpoint columns        : {existing.columns.tolist()}")

sentences_to_embed length : 38972
fls length                : 92458


In [ ]:
log.info("Embedding %d sentences...", len(sentences_to_embed))

embeddings = embedder.encode(
    sentences_to_embed,
    batch_size           = 256,
    show_progress_bar    = True,
    normalize_embeddings = True,   # unit-length: cosine similarity = dot product
    convert_to_numpy     = True,
)

log.info("Embeddings shape: %s", embeddings.shape)
print(f"\u2713 Embeddings computed: {embeddings.shape[0]:,} \u00d7 {embeddings.shape[1]}d")

2026-05-07 03:29:25,660 [INFO] Embedding 38972 sentences...
Batches: 100%|██████████| 153/153 [00:11<00:00, 13.19it/s]
2026-05-07 03:29:37,770 [INFO] Embeddings shape: (38972, 384)


✓ Embeddings computed: 38,972 × 384d


## 6. Value Parser

Matching commitment pairs need their `value` slots compared numerically to determine status (Delivered / Raised / Missed). The value parser handles the common financial figure formats found in the corpus:

- Dollar amounts: `"$4.2 billion"`, `"$500 million"`, `"$1.2B"`
- Percentages: `"18%"`, `"approximately 18%"`, `"up 8%"`
- Ranges: `"$500-600 million"`, `"18% to 20%"` — midpoint is used for comparison
- Bare numbers: `"2.50"`, `"14"`

When parsing fails — for qualitative values like `"a bit higher"` or `"strong growth"` — the parser returns `None` and status assignment falls back to string comparison.

In [ ]:
# Unit multipliers for dollar amount normalisation
UNIT_MULTIPLIERS = {
    "trillion": 1e12, "t": 1e12,
    "billion":  1e9,  "b": 1e9,
    "million":  1e6,  "m": 1e6,
    "thousand": 1e3,  "k": 1e3,
}

# Regex for a single numeric value with optional unit
_NUM_RE = re.compile(
    r"""\$?\s*([0-9]+(?:\.[0-9]+)?)   # numeric part
        \s*(%|                         # percent sign
        (?:trillion|billion|million|thousand|[tbmk]))?  # unit word
    """,
    re.IGNORECASE | re.VERBOSE,
)

# Range separators
_RANGE_RE = re.compile(r"[-\u2013\u2014to]+", re.IGNORECASE)

In [ ]:
def _parse_single(text: str):
    """Parse one numeric value string into a float. Returns None on failure."""
    m = _NUM_RE.search(text)
    if not m:
        return None
    number = float(m.group(1))
    unit   = (m.group(2) or "").lower().strip()
    if unit == "%":
        return number
    multiplier = UNIT_MULTIPLIERS.get(unit, 1.0)
    return number * multiplier

In [ ]:
def parse_value(value_str):
    """
    Parse a financial value string into a comparable float.

    Handles single values, ranges (returns midpoint), and common
    qualifiers (approximately, up, down, more than, less than).
    Returns None when the string cannot be meaningfully parsed.

    Args:
        value_str : Raw value string from Stage 4 slot filling

    Returns:
        Float (normalised to base unit) or None
    """
    if not value_str or not isinstance(value_str, str):
        return None

    # Strip qualifiers that don't affect the numeric comparison
    cleaned = re.sub(
        r"\b(approximately|roughly|about|around|up to|at least|"
        r"more than|less than|nearly|over|under|above|below)\b",
        "", value_str, flags=re.IGNORECASE
    ).strip()

    # Try to find a range (two numbers separated by a range delimiter)
    parts = _RANGE_RE.split(cleaned)
    nums  = []
    for part in parts:
        v = _parse_single(part.strip())
        if v is not None:
            nums.append(v)

    if len(nums) >= 2:
        return (nums[0] + nums[1]) / 2
    elif len(nums) == 1:
        return nums[0]
    return None

In [ ]:
# Quick sanity check on the parser
test_cases = [
    ("$4.2 billion",                   4.2e9),
    ("18%",                            18.0),
    ("$500 million to $600 million",   550e6),
    ("approximately $800 million",     800e6),
    ("up 8%",                          8.0),
    ("a bit higher",                   None),
]

In [ ]:
print("Value parser sanity check:")
all_pass = True
for text, expected in test_cases:
    result = parse_value(text)
    status = "\u2713" if result == expected else "\u2717"
    if result != expected:
        all_pass = False
    print(f"  {status}  {text!r:40} \u2192 {result}  (expected {expected})")

result_msg = "All tests passed ✓" if all_pass else "Some tests failed — review parser"
print(f"\n{result_msg}")

Value parser sanity check:
  ✓  '$4.2 billion'                           → 4200000000.0  (expected 4200000000.0)
  ✓  '18%'                                    → 18.0  (expected 18.0)
  ✓  '$500 million to $600 million'           → 550000000.0  (expected 550000000.0)
  ✓  'approximately $800 million'             → 800000000.0  (expected 800000000.0)
  ✓  'up 8%'                                  → 8.0  (expected 8.0)
  ✓  'a bit higher'                           → None  (expected None)

All tests passed ✓


## 7. Metric Normalisation

Before matching, metric strings are normalised to a canonical form. This prevents trivial mismatches between `"revenue"` and `"revenues"`, `"EPS"` and `"earnings per share"`, `"gross margin"` and `"gross margins"`. Normalisation uses a lookup table of common financial metric aliases.

In [ ]:
# Canonical metric name -> set of aliases
METRIC_ALIASES = {
    "revenue"          : {"revenue", "revenues", "sales", "net sales", "total revenue",
                          "total revenues", "top line"},
    "eps"              : {"eps", "earnings per share", "diluted eps", "adjusted eps",
                          "diluted earnings per share", "adjusted diluted eps"},
    "gross margin"     : {"gross margin", "gross margins", "gross profit margin"},
    "operating margin" : {"operating margin", "operating margins", "operating income margin",
                          "ebit margin", "margin", "margins", "eps shape", "eps growth"},
    "ebitda"           : {"ebitda", "adjusted ebitda", "ebitda margin"},
    "free cash flow"   : {"free cash flow", "fcf", "free cash"},
    "capex"            : {"capex", "capital expenditure", "capital expenditures",
                          "capital spending", "capex spending"},
    "operating income" : {"operating income", "operating profit", "ebit"},
    "net income"       : {"net income", "net profit", "net earnings"},
    "growth"           : {"growth", "growth rate", "revenue growth", "sales growth"},
    "dividend"         : {"dividend", "dividends", "dividend per share", "dps"},
    "buyback"          : {"buyback", "buybacks", "share repurchase", "share repurchases",
                          "repurchase", "stock repurchase"},
}

In [ ]:
# Build reverse lookup: alias -> canonical
_ALIAS_TO_CANONICAL = {}
for canonical, aliases in METRIC_ALIASES.items():
    for alias in aliases:
        _ALIAS_TO_CANONICAL[alias.lower()] = canonical

In [ ]:
def normalise_metric(metric):
    """
    Map a metric string to its canonical form.
    Returns the lowercased original if no alias match is found.
    Returns None if input is None.
    """
    if not metric or not isinstance(metric, str):
        return None
    cleaned = metric.lower().strip()
    return _ALIAS_TO_CANONICAL.get(cleaned, cleaned)

In [ ]:
matchable["metric_norm"] = matchable["metric"].apply(normalise_metric)
fls["metric_norm"]       = fls["metric"].apply(normalise_metric)

print(f"Top 20 normalised metrics:")
print(matchable["metric_norm"].value_counts().head(20).to_string())

Top 20 normalised metrics:
metric_norm
revenue                        4379
operating margin               2002
growth                         1601
eps                            1175
free cash flow                  768
capex                           754
ebitda                          667
gross margin                    660
effective tax rate              342
buyback                         321
tax rate                        299
operating income                250
operating expenses              247
earnings                        236
interest expense                195
adjusted earnings per share     192
organic growth                  168
net interest income             168
organic revenue growth          155
net interest expense            146


## 8. Cross-Quarter Matching

For each company (ticker), commitments from different quarters are compared pairwise using their sentence embeddings. The matching algorithm:

1. Groups the company's commitments by quarter chronologically
2. For each pair of quarters (earlier → later), computes cosine similarity between all commitment pairs
3. Commitment pairs with similarity >= `SIMILARITY_THRESHOLD` and the same normalised metric are marked as matches
4. Each earlier commitment gets at most one match to the highest-similarity later commitment

The matching is done within-company only — cross-company comparison is not meaningful for commitment tracking.

Runtime: approximately 5-15 minutes depending on the number of companies and commitments per company. Companies with many commitments per quarter dominate the runtime since the pairwise comparison is O(n²) within each quarter pair.

In [ ]:
def assign_status(earlier_value, later_value):
    if not isinstance(earlier_value, str):
        earlier_value = None
    if not isinstance(later_value, str):
        later_value = None

    if later_value is None:
        return "Pending"

    # Attempt numeric comparison
    earlier_num = parse_value(earlier_value)
    later_num   = parse_value(later_value)

    if earlier_num is not None and later_num is not None and earlier_num != 0:
        ratio = later_num / earlier_num
        if ratio > 1.05:
            return "Raised"
        elif ratio >= 0.95:
            return "Delivered"
        else:
            return "Missed"

    # Fuzzy string fallback for non-parseable values
    if earlier_value and later_value:
        similarity = SequenceMatcher(
            None,
            earlier_value.lower().strip(),
            later_value.lower().strip()
        ).ratio()
        if similarity >= 0.80:
            return "Delivered"
        # Can't meaningfully compare — leave as Pending
        return "Pending"

    return "Pending"

In [ ]:
# Build index: commitment_id -> row index in matchable for fast embedding lookup
matchable = matchable.reset_index(drop=True)
id_to_idx = {row["commitment_id"]: i for i, row in matchable.iterrows()}

# Results: list of (commitment_id, matched_commitment_id, status)
match_results = []
tickers = matchable["ticker"].unique()

In [ ]:
log.info("Running cross-quarter matching for %d companies...", len(tickers))

for ticker in tqdm(tickers, desc="Cross-quarter matching"):
    company_df = matchable[matchable["ticker"] == ticker].copy()

    # Need at least 2 distinct quarters to do any matching
    quarters = sorted(company_df["quarter_key"].unique())
    if len(quarters) < 2:
        continue

    # Compare each earlier quarter against each later quarter
    for q_earlier, q_later in combinations(quarters, 2):
        earlier_df = company_df[company_df["quarter_key"] == q_earlier]
        later_df   = company_df[company_df["quarter_key"] == q_later]

        if earlier_df.empty or later_df.empty:
            continue

        # Get embedding vectors for each group
        e_indices = [id_to_idx[cid] for cid in earlier_df["commitment_id"]]
        l_indices = [id_to_idx[cid] for cid in later_df["commitment_id"]]

        e_embs = embeddings[e_indices]   # shape: (n_earlier, 384)
        l_embs = embeddings[l_indices]   # shape: (n_later, 384)

        # Pairwise cosine similarity matrix
        sim_matrix = cosine_similarity(e_embs, l_embs)  # (n_earlier, n_later)

        # For each earlier commitment, find best matching later commitment
        for e_pos, e_row in enumerate(earlier_df.itertuples()):
            best_sim   = -1
            best_l_pos = -1
            best_l_row = None

            for l_pos, l_row in enumerate(later_df.itertuples()):
                sim = sim_matrix[e_pos, l_pos]
                if sim < SIMILARITY_THRESHOLD:
                    continue

                # Metric must match (if both present) to avoid false positives
                e_metric = e_row.metric_norm
                l_metric = l_row.metric_norm
                if e_metric and l_metric and e_metric != l_metric and sim < 0.75:
                    continue

                if sim > best_sim:
                    best_sim   = sim
                    best_l_pos = l_pos
                    best_l_row = l_row

            if best_l_pos >= 0:
                status = assign_status(
                    earlier_value = e_row.value,
                    later_value   = best_l_row.value,
                )
                match_results.append({
                    "commitment_id"        : e_row.commitment_id,
                    "matched_commitment_id": best_l_row.commitment_id,
                    "status"               : status,
                    "similarity"           : round(best_sim, 4),
                })

log.info("Matching complete \u2014 %d matched pairs found", len(match_results))
print(f"\nMatched pairs found : {len(match_results):,}")

2026-05-07 03:29:39,279 [INFO] Running cross-quarter matching for 531 companies...
Cross-quarter matching: 100%|██████████| 531/531 [04:34<00:00,  1.93it/s]
2026-05-07 03:34:13,731 [INFO] Matching complete — 41703 matched pairs found



Matched pairs found : 41,703


## 9. Results & Validation

The matched pairs and their assigned statuses are examined. Expected distribution based on financial guidance literature:

- **Delivered** — 40-55% (management typically meets guidance)
- **Raised** — 15-25% (upward revisions are common in strong markets)
- **Missed** — 15-25% (misses are less common but highly informative)
- **Withdrawn** — tracked separately as commitments with no subsequent match

Sample matched pairs are printed for qualitative validation — the earlier and later sentences should clearly be the same commitment in different words.

In [ ]:
matches_df = pd.DataFrame(match_results)

print("=" * 52)
print("  EarningsLens \u2014 Stage 6 Matching Results")
print("=" * 52)
print(f"\nTotal matched pairs      : {len(matches_df):,}")
print(f"Unique earlier commits    : {matches_df['commitment_id'].nunique():,}")
print(f"\nStatus distribution:")
status_counts = matches_df["status"].value_counts()
for status, count in status_counts.items():
    pct = count / len(matches_df) * 100
    print(f"  {status:<12}: {count:>7,}  ({pct:.1f}%)")

print(f"\nSimilarity score distribution:")
print(matches_df["similarity"].describe().round(3).to_string())

  EarningsLens — Stage 6 Matching Results

Total matched pairs      : 41,703
Unique earlier commits    : 16,180

Status distribution:
  Raised      :  12,950  (31.1%)
  Delivered   :  12,082  (29.0%)
  Missed      :  10,531  (25.3%)
  Pending     :   6,140  (14.7%)

Similarity score distribution:
count    41703.000
mean         0.857
std          0.084
min          0.700
25%          0.786
50%          0.851
75%          0.930
max          1.000


In [ ]:
# Withdrawn commitments: earlier commits with no match in any later quarter
matched_earlier_ids = set(matches_df["commitment_id"].tolist())

all_earlier_ids = set()
for ticker in matchable["ticker"].unique():
    company_df = matchable[matchable["ticker"] == ticker]
    quarters   = sorted(company_df["quarter_key"].unique())
    if len(quarters) < 2:
        continue
    # All quarters except the last are candidates for withdrawal
    for q in quarters[:-1]:
        ids = company_df[company_df["quarter_key"] == q]["commitment_id"].tolist()
        all_earlier_ids.update(ids)

In [ ]:
withdrawn_ids = all_earlier_ids - matched_earlier_ids
print(f"Withdrawn commitments    : {len(withdrawn_ids):,}")
print(f"  (commitments from non-final quarters with no subsequent match)")

Withdrawn commitments    : 18,556
  (commitments from non-final quarters with no subsequent match)


In [ ]:
# Print sample matched pairs for qualitative validation
print("\u2500\u2500 Sample matched commitment pairs \u2500\u2500\n")

sample_matches = matches_df.sample(min(6, len(matches_df)), random_state=42)

for _, match in sample_matches.iterrows():
    e_row = matchable[matchable["commitment_id"] == match["commitment_id"]].iloc[0]
    l_row = matchable[matchable["commitment_id"] == match["matched_commitment_id"]].iloc[0]

    print(f"  Ticker    : {e_row['ticker']}")
    print(f"  Status    : {match['status']}  (similarity: {match['similarity']})")
    print(f"  Earlier   : [{e_row['year']} Q{e_row['quarter']}] {str(e_row['sentence'])[:110]}")
    print(f"    metric={e_row['metric']}  value={e_row['value']}  timeframe={e_row['timeframe']}")
    print(f"  Later     : [{l_row['year']} Q{l_row['quarter']}] {str(l_row['sentence'])[:110]}")
    print(f"    metric={l_row['metric']}  value={l_row['value']}  timeframe={l_row['timeframe']}")
    print()

── Sample matched commitment pairs ──

  Ticker    : TJX
  Status    : Delivered  (similarity: 0.8235999941825867)
  Earlier   : [2024 Q4] We're expecting first quarter SG&A to be approximately 19.5%, up 50 basis points versus last year.
    metric=SG&A as percentage  value=approximately 19.5%  timeframe=Q1
  Later     : [2025 Q2] We continue to expect SG&A to be approximately 19.3%, flat versus last year's 19.3%.
    metric=SG&A as a percentage of revenue  value=approximately 19.3%  timeframe=flat versus last year

  Ticker    : GE
  Status    : Pending  (similarity: 0.8137000203132629)
  Earlier   : [2024 Q3] We expect DPT to be at the lower end of the current profit range of $1 billion to $1.3 billion.
    metric=profit  value=at the lower end of the current profit range of $1 billion to $1.3 billion  timeframe=nan
  Later     : [2024 Q4] In DPT, we expect mid- to high-single digits revenue growth with increased defense units and profit in the ran
    metric=revenue  value=mid- to h

## 10. Write Results - Local Parquet and Supabase

The matched pair results are written back to `fls` via `commitment_id`. Three columns are updated:

- `status` — updated from `Pending` to `Delivered`, `Raised`, `Missed`, or `Withdrawn` for matched commitments
- `matched_commitment_id` — the `commitment_id` of the matched later commitment
- `match_similarity` — the cosine similarity score of the match

Unmatched commitments in the final quarter retain `status = Pending`. Withdrawn commitments (earlier quarters with no match) are updated to `Withdrawn`.

In [ ]:
# Add match_similarity column
if "match_similarity" not in fls.columns:
    fls["match_similarity"] = None

In [ ]:
# Build lookup from matches_df for fast update
match_lookup = {
    row["commitment_id"]: {
        "status"               : row["status"],
        "matched_commitment_id": row["matched_commitment_id"],
        "match_similarity"     : row["similarity"],
    }
    for _, row in matches_df.iterrows()
}


In [ ]:
# Apply matched status and IDs
for cid, update in match_lookup.items():
    mask = fls["commitment_id"] == cid
    fls.loc[mask, "status"]                = update["status"]
    fls.loc[mask, "matched_commitment_id"] = update["matched_commitment_id"]
    fls.loc[mask, "match_similarity"]      = update["match_similarity"]

In [ ]:
# Apply Withdrawn status
withdrawn_mask = fls["commitment_id"].isin(withdrawn_ids)
fls.loc[withdrawn_mask & (fls["status"] == "Pending"), "status"] = "Withdrawn"

In [ ]:
# Save updated parquet
fls.to_parquet(OUTPUT_PATH, index=False)

print("\u2713 fls_2023_2025.parquet updated with Stage 6 tracking results")
print(f"\nFinal status distribution:")
status_dist = fls["status"].value_counts()
for status, count in status_dist.items():
    pct = count / len(fls) * 100
    print(f"  {status:<12}: {count:>8,}  ({pct:.1f}%)")

✓ fls_2023_2025.parquet updated with Stage 6 tracking results

Final status distribution:
  Pending     :   60,776  (65.7%)
  Withdrawn   :   18,556  (20.1%)
  Raised      :    4,767  (5.2%)
  Delivered   :    4,213  (4.6%)
  Missed      :    4,146  (4.5%)


In [ ]:
conn   = psycopg2.connect(DATABASE_URL + "?sslmode=require")
cursor = conn.cursor()

def safe_matched_id(val):
    if pd.isna(val):
        return None
    return int(val)

status_rows = [
    (row["status"], safe_matched_id(row.get("matched_commitment_id")), row["sentence_hash"])
    for _, row in fls.iterrows()
    if pd.notna(row.get("status"))
]

execute_values(
    cursor,
    """
    UPDATE commitments
    SET status                = data.status,
        matched_commitment_id = data.matched_commitment_id::int
    FROM (VALUES %s) AS data(status, matched_commitment_id, sentence_hash)
    WHERE commitments.sentence_hash = data.sentence_hash
    """,
    status_rows,
    page_size=2000
)
conn.commit()
conn.close()
print(f"✓ Stage 6 — {len(status_rows):,} statuses written to Supabase")

✓ Stage 6 — 92,458 statuses written to Supabase


## 11. Credibility Summary per Company

A company-level credibility summary is computed from the matched commitment statuses. This summary table is what the Stage 10 dashboard displays in the guidance delta table — the primary analytical output of EarningsLens.

The credibility score per company is defined as:
```
credibility = (Delivered + Raised) / (Delivered + Raised + Missed)
```
This is the fraction of resolved commitments that were met or exceeded, excluding Withdrawn and Pending commitments from the denominator.

In [ ]:
# Compute per-company credibility scores from matched commitments only
resolved_statuses = {"Delivered", "Raised", "Missed"}
resolved = fls[fls["status"].isin(resolved_statuses)]

In [ ]:

credibility_rows = []
for ticker, group in resolved.groupby("ticker"):
    counts      = group["status"].value_counts().to_dict()
    delivered   = counts.get("Delivered", 0)
    raised      = counts.get("Raised", 0)
    missed      = counts.get("Missed", 0)
    total       = delivered + raised + missed
    credibility = (delivered + raised) / total if total > 0 else None
    mean_hedge  = group["hedge_score"].mean() if "hedge_score" in group.columns else None

    credibility_rows.append({
        "ticker"         : ticker,
        "company_name"   : group["company_name"].iloc[0],
        "delivered"      : delivered,
        "raised"         : raised,
        "missed"         : missed,
        "total_resolved" : total,
        "credibility"    : round(credibility, 3) if credibility else None,
        "mean_hedge"     : round(mean_hedge, 3) if mean_hedge else None,
    })

In [ ]:
credibility_df = pd.DataFrame(credibility_rows)
credibility_df = credibility_df.sort_values("credibility", ascending=False)

# Save credibility summary as a separate file for dashboard use
credibility_df.to_parquet("credibility_summary.parquet", index=False)

print("=" * 52)
print("  Company Credibility Summary")
print("=" * 52)
print(f"\nCompanies with resolved commitments : {len(credibility_df):,}")
print(f"\nTop 15 most credible companies:")
print(credibility_df.head(15)[["ticker", "company_name", "delivered", "raised",
                               "missed", "credibility", "mean_hedge"]].to_string(index=False))
print(f"\nBottom 10 least credible companies:")
print(credibility_df.tail(10)[["ticker", "company_name", "delivered", "raised",
                               "missed", "credibility", "mean_hedge"]].to_string(index=False))

  Company Credibility Summary

Companies with resolved commitments : 500

Top 15 most credible companies:
ticker                    company_name  delivered  raised  missed  credibility  mean_hedge
   YUM               Yum! Brands, Inc.          5       3       0          1.0       0.397
   WBD    Warner Bros. Discovery, Inc.          2       2       0          1.0       0.239
   XEL                Xcel Energy Inc.          3       1       0          1.0       0.492
   UNP       Union Pacific Corporation          1       0       0          1.0       0.175
  VLTO             Veralto Corporation          8       3       0          1.0       0.380
  MKTX       MarketAxess Holdings Inc.          0       6       0          1.0       0.536
   MLM Martin Marietta Materials, Inc.          5       3       0          1.0       0.308
   AMP      Ameriprise Financial, Inc.          2       0       0          1.0       0.473
  ULTA               Ulta Beauty, Inc.          1       0       0          

In [ ]:
credibility_df = pd.read_parquet("credibility_summary.parquet")
credibility_df.head()

,ticker,company_name,delivered,raised,missed,total_resolved,credibility,mean_hedge
0,YUM,"Yum! Brands, Inc.",5,3,0,8,1.0,0.397
1,WBD,"Warner Bros. Discovery, Inc.",2,2,0,4,1.0,0.239
2,XEL,Xcel Energy Inc.,3,1,0,4,1.0,0.492
3,UNP,Union Pacific Corporation,1,0,0,1,1.0,0.175
4,VLTO,Veralto Corporation,8,3,0,11,1.0,0.380


In [ ]:
conn = psycopg2.connect(DATABASE_URL + "?sslmode=require")
cursor = conn.cursor()

# Prepare rows
summary_rows = [
    (
        row["ticker"],
        row["company_name"],
        int(row["delivered"]) if pd.notna(row["delivered"]) else 0,
        int(row["raised"]) if pd.notna(row["raised"]) else 0,
        int(row["missed"]) if pd.notna(row["missed"]) else 0,
        int(row["total_resolved"]) if pd.notna(row["total_resolved"]) else 0,
        float(row["credibility"]) if pd.notna(row["credibility"]) else None,
        float(row["mean_hedge"]) if pd.notna(row["mean_hedge"]) else None,
    )
    for _, row in credibility_df.iterrows()
]

execute_values(
    cursor,
    """
    INSERT INTO credibility_summary (
        ticker,
        company_name,
        delivered,
        raised,
        missed,
        total_resolved,
        credibility,
        mean_hedge
    )
    VALUES %s
    """,
    summary_rows,
    page_size=2000
)

conn.commit()
conn.close()

print(f"✓ credibility_summary — {len(summary_rows):,} rows written to Supabase")

✓ credibility_summary — 500 rows written to Supabase


## Stage 6 Summary

In [ ]:
print("=" * 52)
print("  EarningsLens \u2014 Stage 6 Complete")
print("=" * 52)
print(f"\nCompanies tracked        : {fls['ticker'].nunique():,}")
print(f"Total commitments        : {len(fls):,}")
print(f"Matched pairs            : {len(matches_df):,}")
print(f"Withdrawn commitments    : {len(withdrawn_ids):,}")
print(f"\nStatus breakdown:")
for status, count in fls["status"].value_counts().items():
    print(f"  {status:<12}: {count:>8,}")
print(f"\nCredibility summary saved : credibility_summary.parquet")
print(f"Commitments saved         : fls_2023_2025.parquet")
print(f"\n\u2192 Ready for Stage 7 \u2014 Sentiment Analysis")

  EarningsLens — Stage 6 Complete

Companies tracked        : 531
Total commitments        : 92,458
Matched pairs            : 41,703
Withdrawn commitments    : 18,556

Status breakdown:
  Pending     :   60,776
  Withdrawn   :   18,556
  Raised      :    4,767
  Delivered   :    4,213
  Missed      :    4,146

Credibility summary saved : credibility_summary.parquet
Commitments saved         : fls_2023_2025.parquet

→ Ready for Stage 7 — Sentiment Analysis


: 

## Handoff to Stage 7

Stage 6 is complete. The outputs are:

| Output | Location | Consumed by |
|---|---|---|
| `fls_2023_2025.parquet` | Local disk | Stage 7, 8, 9, 10 |
| `credibility_summary.parquet` | Local disk | Stage 9 (agentic), Stage 10 (dashboard) |

`fls_2023_2025.parquet` now carries the complete pipeline output - `fls_label`, `fls_confidence`, `metric`, `value`, `timeframe`, `hedge_score`, `status`, `matched_commitment_id`, and `match_similarity` for all 269,679 Specific FLS sentences in the 2023-2025 scope.

Stage 7 computes FinBERT and VADER sentiment scores across the full transcript text (not just Specific FLS sentences) to provide supporting tone context for the dashboard. It operates on `sentences_df` from Stage 2, specifically the executive prepared_remarks segments and writes aggregate sentiment scores per transcript to a separate `sentiment_scores.parquet` file.